# MicroNIR: Ultra-Lightweight RGB to NIR Reconstruction

Spectral reconstruction from RGB to NIR/Red-Edge (Sentinel-2) using an ultra-compact architecture.

## Architecture
1. **Shallow Feature Extraction** - Single Conv 3×3 → 64ch
2. **6× Inverted Residual Blocks (BSConv)** - Expand 1×1 → Depthwise 3×3 → Project 1×1
3. **Squeeze-and-Excitation (SE)** - Global channel recalibration
4. **Convolutional Decoder** - Refine + project to 4 bands (B5, B6, B7, B8)
5. **Global Residual** - RGB→NIR linear projection as baseline

## Key Features
- Depthwise Separable Convolutions (MobileNet-style)
- Inverted Residual Blocks (MobileNetV2) with expansion ratio 2
- SE attention for channel recalibration at negligible cost
- Mixed Precision (fp16) + EMA for stable training
- Designed for real-time drone inference

## 1. Configuration and Imports

In [ ]:
import tensorflow as tf
import numpy as np
import math
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

DRIVE_SAVE_PATH = '/content/drive/MyDrive/MicroNIR_Checkpoints'
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

print(f"Google Drive mounted")
print(f"Checkpoint folder: {DRIVE_SAVE_PATH}")

def sync_to_drive(local_path, drive_path=DRIVE_SAVE_PATH):
    """Copy file or folder to Google Drive."""
    if os.path.isfile(local_path):
        dst = os.path.join(drive_path, os.path.basename(local_path))
        shutil.copy2(local_path, dst)
        print(f"  Copied to Drive: {os.path.basename(local_path)}")
    elif os.path.isdir(local_path):
        dst = os.path.join(drive_path, os.path.basename(local_path))
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(local_path, dst)
        print(f"  Folder copied to Drive: {os.path.basename(local_path)}")

def restore_from_drive(drive_path=DRIVE_SAVE_PATH, local_path='.'):
    """Restore checkpoints from Drive."""
    if os.path.exists(drive_path):
        for item in os.listdir(drive_path):
            src = os.path.join(drive_path, item)
            dst = os.path.join(local_path, item)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
                print(f"  Restored: {item}")
            elif os.path.isdir(src):
                if os.path.exists(dst):
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
                print(f"  Folder restored: {item}")
        print("Restoration complete!")
    else:
        print("No backup found on Drive")

print("\nTo restore after disconnection, run:")
print("   restore_from_drive()")

### Deep Learning Utilities

| Technique | Purpose |
|-----------|--------|
| **DropPath** | Regularization, random block drop |
| **EMA** | Exponential Moving Average of weights for stable models |
| **Gradient Clipping** | Prevents gradient explosion |
| **LR Warmup + Cosine** | Progressive LR increase then decay |
| **RMSNorm** | Faster than LayerNorm, often better |

In [ ]:
import math

def get_activation(name):
    """Returns activation name for Keras 3."""
    activations = {
        'relu': 'relu',
        'gelu': 'gelu',
        'swish': 'swish',
        'silu': 'swish',
        'mish': 'mish',
    }
    if name.lower() not in activations:
        raise ValueError(f"Unknown activation: {name}. Options: {list(activations.keys())}")
    return activations[name.lower()]


def get_activation_layer(name):
    """Returns a Keras Activation layer."""
    activation_name = get_activation(name)
    return layers.Activation(activation_name)


def get_initializer(activation='relu'):
    """Returns appropriate initializer based on activation."""
    if activation in ['relu', 'leaky_relu', 'elu']:
        return tf.keras.initializers.HeNormal()
    else:
        return tf.keras.initializers.GlorotUniform()


class DropPath(layers.Layer):
    """Stochastic Depth / DropPath (Huang et al., 2016)"""
    def __init__(self, drop_prob=0.1, **kwargs):
        super().__init__(**kwargs)
        self.drop_prob = drop_prob

    def call(self, x, training=None):
        if not training or self.drop_prob == 0:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape) - 1)
        random_tensor = keep_prob + tf.random.uniform(shape, dtype=x.dtype)
        binary_mask = tf.floor(random_tensor)
        return x / keep_prob * binary_mask

    def get_config(self):
        config = super().get_config()
        config.update({'drop_prob': self.drop_prob})
        return config


class RMSNorm(layers.Layer):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, 2019)"""
    def __init__(self, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def build(self, input_shape):
        self.gamma = self.add_weight(
            name='gamma',
            shape=(input_shape[-1],),
            initializer='ones',
            trainable=True
        )

    def call(self, x):
        x_float = tf.cast(x, tf.float32)
        rms = tf.sqrt(tf.reduce_mean(tf.square(x_float), axis=-1, keepdims=True) + self.eps)
        x_normed = x_float / rms
        gamma = tf.cast(self.gamma, x_float.dtype)
        result = x_normed * gamma
        return tf.cast(result, x.dtype)

    def get_config(self):
        config = super().get_config()
        config.update({'eps': self.eps})
        return config


class EMA:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.9999):
        self.model = model
        self.decay = decay
        self.ema_weights = [tf.Variable(w, trainable=False) for w in model.weights]

    def update(self):
        for ema_w, model_w in zip(self.ema_weights, self.model.weights):
            ema_w.assign(self.decay * ema_w + (1 - self.decay) * model_w)

    def apply_to_model(self):
        self.backup_weights = [tf.Variable(w) for w in self.model.weights]
        for model_w, ema_w in zip(self.model.weights, self.ema_weights):
            model_w.assign(ema_w)

    def restore_model(self):
        for model_w, backup_w in zip(self.model.weights, self.backup_weights):
            model_w.assign(backup_w)


class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Learning Rate Schedule with Warmup + Cosine Decay."""
    def __init__(self, base_lr, warmup_steps, decay_steps, warmup_min_lr=1e-6):
        super().__init__()
        self.base_lr = base_lr
        self.warmup_steps = warmup_steps
        self.decay_steps = decay_steps
        self.warmup_min_lr = warmup_min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        total_steps = tf.cast(self.decay_steps, tf.float32)

        warmup_progress = step / tf.maximum(warmup_steps, 1.0)
        warmup_lr = self.warmup_min_lr + (self.base_lr - self.warmup_min_lr) * warmup_progress

        decay_steps = total_steps - warmup_steps
        cosine_progress = (step - warmup_steps) / tf.maximum(decay_steps, 1.0)
        cosine_decay = 0.5 * (1 + tf.cos(math.pi * cosine_progress))
        decay_lr = self.warmup_min_lr + (self.base_lr - self.warmup_min_lr) * cosine_decay

        return tf.where(step < warmup_steps, warmup_lr, decay_lr)

    def get_config(self):
        return {
            'base_lr': self.base_lr,
            'warmup_steps': self.warmup_steps,
            'decay_steps': self.decay_steps,
            'warmup_min_lr': self.warmup_min_lr
        }

print("Advanced utilities defined")

In [ ]:
class Config:
    """MicroNIR configuration."""

    # GCS Data
    BUCKET = 'YOUR_BUCKET'
    FOLDER = 'dataset_training_mst'
    PATCH_SIZE = 128
    INPUT_CHANNELS = 3
    OUTPUT_CHANNELS = 4

    # Output constraint
    OUTPUT_CONSTRAINT = "tanh"
    OUTPUT_MIN = 0.0
    OUTPUT_MAX = 1.0

    # Architecture - Ultra Lightweight
    BASE_DIM = 64             # Base channel width
    NUM_BLOCKS = 6            # Number of Inverted Residual blocks
    EXPANSION_RATIO = 2       # Expansion in Inverted Residual
    SE_RATIO = 4              # Squeeze-Excitation reduction ratio
    DROPOUT = 0.05            # Light dropout (small model)
    DROP_PATH_RATE = 0.1      # Stochastic depth

    # Training
    BATCH_SIZE = 32
    EPOCHS = 30
    LEARNING_RATE = 1e-3      # Higher LR for small model
    WEIGHT_DECAY = 1e-4

    WARMUP_STEPS = 500
    WARMUP_MIN_LR = 1e-5

    # Loss weights (L1 + SAM)
    LAMBDA_L1 = 1.0
    LAMBDA_SAM = 0.5
    SAM_MODE = "cosine"
    SAM_LOSS_SCALE = 1.0

    # Per-band L1 weights (B5, B6, B7, B8)
    USE_BAND_L1_WEIGHTS = True
    BAND_L1_WEIGHTS = [1.5, 1.0, 1.0, 2.0]

    # Stability
    USE_MIXED_PRECISION = True
    GRADIENT_CLIP_NORM = 1.0
    USE_EMA = True
    EMA_DECAY = 0.999

    # Activation
    ACTIVATION = 'gelu'

    # Drive checkpointing
    SAVE_TO_DRIVE = True
    DRIVE_PATH = '/content/drive/MyDrive/MicroNIR_Checkpoints'

config = Config()

print("Configuration MicroNIR v1")
print(f"  Architecture:")
print(f"    - BASE_DIM: {config.BASE_DIM}")
print(f"    - NUM_BLOCKS: {config.NUM_BLOCKS}")
print(f"    - EXPANSION_RATIO: {config.EXPANSION_RATIO}")
print(f"    - SE_RATIO: {config.SE_RATIO}")
print(f"  Loss: L1 + SAM")
print(f"  Training:")
print(f"    - BATCH_SIZE: {config.BATCH_SIZE}")
print(f"    - LEARNING_RATE: {config.LEARNING_RATE}")
print(f"    - MIXED PRECISION: {config.USE_MIXED_PRECISION}")

## 2. MicroNIR Architecture

Ultra-lightweight design based on proven mobile architectures:

| Component | Technique | Why |
|-----------|-----------|-----|
| **Inverted Residual** | MobileNetV2 | Expand→DW→Project with skip = max efficiency |
| **Depthwise Separable Conv** | MobileNet | 8-9× fewer params than standard conv |
| **SE Attention** | SENet | Channel recalibration at ~500 params cost |
| **Global Residual** | ResNet | RGB→NIR linear projection as baseline |
| **RMSNorm** | T5/LLaMA | Faster than LayerNorm, equally effective |

```
RGB (3ch) → Conv3×3(64) → 6× InvRes(64, exp=2) → SE(64) → Refine → NIR (4ch)
    │                                                                    ↑
    └──────────── Conv1×1 (global residual) ─────────────────────────────┘
```

**Target: ~130K parameters** (vs ~1M for CNN_MSGPP)

In [ ]:
class SqueezeExcitation(layers.Layer):
    """Squeeze-and-Excitation block (Hu et al., 2018).

    Global channel attention at negligible parameter cost.
    For 64 channels with ratio=4: only 2×(64×16 + 16) = 2,080 params.
    """
    def __init__(self, channels, ratio=4, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.ratio = ratio

    def build(self, input_shape):
        reduced = max(self.channels // self.ratio, 4)
        self.fc1 = layers.Dense(reduced, kernel_initializer='he_normal')
        self.fc2 = layers.Dense(self.channels, kernel_initializer='glorot_normal')
        super().build(input_shape)

    def call(self, x):
        # Global Average Pooling
        gap = tf.reduce_mean(x, axis=[1, 2], keepdims=False)  # (B, C)
        # Squeeze
        s = self.fc1(gap)
        s = tf.nn.swish(s)
        # Excitation
        s = self.fc2(s)
        s = tf.nn.sigmoid(s)
        # Scale
        s = tf.reshape(s, (-1, 1, 1, self.channels))
        return x * s

    def get_config(self):
        config = super().get_config()
        config.update({'channels': self.channels, 'ratio': self.ratio})
        return config


class InvertedResidualBlock(layers.Layer):
    """Inverted Residual Block (MobileNetV2 + BSConv style).

    Pattern: Expand (1×1) → Depthwise (3×3) → Project (1×1) + Skip

    For channels=64, expansion=2:
      - Expand: 64→128 (1×1) = 8,320 params
      - DWConv: 128ch (3×3) = 1,280 params
      - Project: 128→64 (1×1) = 8,256 params
      - Total: ~18K params (vs ~74K for standard 2× Conv3×3)
    """
    def __init__(self, channels, expansion=2, drop_path_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.expansion = expansion
        self.drop_path_rate = drop_path_rate

    def build(self, input_shape):
        expanded = self.channels * self.expansion

        # Pre-norm
        self.norm = RMSNorm(name='pre_norm')

        # Expand (pointwise)
        self.expand_conv = layers.Conv2D(
            expanded, 1, padding='same',
            kernel_initializer='he_normal',
            use_bias=False, name='expand'
        )
        self.expand_norm = RMSNorm(name='expand_norm')
        self.expand_act = layers.Activation('gelu', name='expand_act')

        # Depthwise spatial mixing
        self.dwconv = layers.DepthwiseConv2D(
            3, padding='same',
            depthwise_initializer='he_normal',
            use_bias=False, name='dwconv'
        )
        self.dw_norm = RMSNorm(name='dw_norm')
        self.dw_act = layers.Activation('gelu', name='dw_act')

        # Project (pointwise) - no activation after projection
        self.project_conv = layers.Conv2D(
            self.channels, 1, padding='same',
            kernel_initializer='glorot_uniform',
            use_bias=False, name='project'
        )

        # DropPath for regularization
        self.drop_path = DropPath(self.drop_path_rate) if self.drop_path_rate > 0 else None

        super().build(input_shape)

    def call(self, x, training=False):
        residual = x

        # Pre-norm → Expand → DWConv → Project
        y = self.norm(x)
        y = self.expand_conv(y)
        y = self.expand_norm(y)
        y = self.expand_act(y)

        y = self.dwconv(y)
        y = self.dw_norm(y)
        y = self.dw_act(y)

        y = self.project_conv(y)

        # DropPath + Skip connection
        if self.drop_path is not None:
            y = self.drop_path(y, training=training)

        return residual + y

    def get_config(self):
        config = super().get_config()
        config.update({
            'channels': self.channels,
            'expansion': self.expansion,
            'drop_path_rate': self.drop_path_rate,
        })
        return config


def build_micronir(config):
    """Build MicroNIR: ultra-lightweight RGB → NIR model.

    Architecture:
      RGB → Stem(Conv3×3) → N× InvRes → SE → Refine → Output
       │                                                  ↑
       └──────── Conv1×1 (global residual) ──────────────┘
    """
    inputs = keras.Input(shape=(config.PATCH_SIZE, config.PATCH_SIZE, config.INPUT_CHANNELS))

    init = get_initializer(config.ACTIVATION)
    activation_name = get_activation(config.ACTIVATION)

    # ======== Global Skip: Direct RGB → NIR projection ========
    rgb_to_nir = layers.Conv2D(
        config.OUTPUT_CHANNELS, 1,
        padding='same',
        use_bias=True,
        kernel_initializer='glorot_uniform',
        bias_initializer=tf.keras.initializers.Constant(0.1),
        dtype='float32',
        name='rgb_to_nir_proj'
    )(inputs)

    # ======== Shallow Feature Extraction (Stem) ========
    x = layers.Conv2D(
        config.BASE_DIM, 3, padding='same',
        kernel_initializer=init, name='stem_conv'
    )(inputs)
    x = RMSNorm(name='stem_norm')(x)
    x = layers.Activation(activation_name, name='stem_act')(x)

    # ======== Inverted Residual Blocks ========
    num_blocks = config.NUM_BLOCKS
    dpr = [i * config.DROP_PATH_RATE / max(num_blocks - 1, 1) for i in range(num_blocks)]

    for i in range(num_blocks):
        x = InvertedResidualBlock(
            channels=config.BASE_DIM,
            expansion=config.EXPANSION_RATIO,
            drop_path_rate=dpr[i],
            name=f'invres_block_{i}'
        )(x)

    # ======== Post-blocks normalization ========
    x = RMSNorm(name='final_norm')(x)

    # ======== Squeeze-and-Excitation (global channel recalibration) ========
    x = SqueezeExcitation(
        config.BASE_DIM,
        ratio=config.SE_RATIO,
        name='se_block'
    )(x)

    # ======== Lightweight Decoder ========
    x = layers.Conv2D(
        32, 3, padding='same',
        kernel_initializer=init, name='refine_conv'
    )(x)
    x = RMSNorm(name='refine_norm')(x)
    x = layers.Activation(activation_name, name='refine_act')(x)

    # ======== Residual Output ========
    residual = layers.Conv2D(
        config.OUTPUT_CHANNELS, 3,
        padding='same',
        kernel_initializer=tf.keras.initializers.TruncatedNormal(stddev=0.01),
        bias_initializer=tf.keras.initializers.Constant(-0.6689549),
        dtype='float32',
        name='residual_output'
    )(x)

    # Output = RGB projection + learned residual
    outputs = layers.Add(name='final_output')([rgb_to_nir, residual])

    # Output constraint
    output_constraint = getattr(config, 'OUTPUT_CONSTRAINT', None)
    if output_constraint == 'clip':
        outputs = layers.Lambda(
            lambda t: tf.clip_by_value(t, config.OUTPUT_MIN, config.OUTPUT_MAX),
            name='output_clip'
        )(outputs)
    elif output_constraint == 'tanh':
        outputs = layers.Lambda(
            lambda t: (0.5 * (tf.tanh(t) + 1.0)) * (config.OUTPUT_MAX - config.OUTPUT_MIN) + config.OUTPUT_MIN,
            name='output_tanh'
        )(outputs)

    model = keras.Model(inputs=inputs, outputs=outputs, name='MicroNIR_v1')
    return model


# ======== Test model ========
print("Testing MicroNIR_v1...")

tf.keras.mixed_precision.set_global_policy('float32')

test_model = build_micronir(config)
param_count = test_model.count_params()
test_model.summary()
print(f"\nMicroNIR_v1 built: {param_count:,} parameters")
print(f"Comparison: CNN_MSGPP = 982,500 params -> {param_count/982500:.1f}x smaller")

dummy_input = np.random.rand(1, 128, 128, 3).astype(np.float32) * 0.2 + 0.02
dummy_output = test_model.predict(dummy_input, verbose=0)

print(f"  Input:  min={dummy_input.min():.4f}, max={dummy_input.max():.4f}, mean={dummy_input.mean():.4f}")
print(f"  Output: min={dummy_output.min():.4f}, max={dummy_output.max():.4f}, mean={dummy_output.mean():.4f}")

if np.isnan(dummy_output).any():
    print("  NaN DETECTED!")
elif 0.05 < dummy_output.mean() < 0.5:
    print("  Output OK!")
else:
    print(f"  Output outside expected range")

del test_model

if config.USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed Precision enabled")

## 3. Loss Functions

Combined loss:

| Domain | Loss | What it captures |
|--------|------|------------------|
| Pixel | L1 (MAE) | Accurate value reconstruction |
| Spectral | SAM | Physically plausible spectral signature |

In [ ]:
def l1_loss_masked(y_true, y_pred, mask, band_weights=None):
    """L1 Loss with validity mask."""
    mask_expanded = mask[..., tf.newaxis]
    abs_diff = tf.abs(y_true - y_pred)

    if band_weights is not None:
        weights = tf.cast(tf.reshape(band_weights, (1, 1, 1, -1)), abs_diff.dtype)
        weights = weights / tf.reduce_mean(weights)
        abs_diff = abs_diff * weights

    masked_diff = abs_diff * mask_expanded
    num_valid = (
        tf.reduce_sum(mask) * tf.cast(tf.shape(abs_diff)[-1], abs_diff.dtype) + 1e-8
    )
    return tf.reduce_sum(masked_diff) / num_valid


def spectral_angle_mapper_loss_masked(y_true, y_pred, mask):
    """SAM Loss with validity mask."""
    eps = 1e-7

    norm_true = tf.sqrt(tf.reduce_sum(tf.square(y_true), axis=-1) + eps)
    norm_pred = tf.sqrt(tf.reduce_sum(tf.square(y_pred), axis=-1) + eps)

    y_true_norm = y_true / (norm_true[..., tf.newaxis] + eps)
    y_pred_norm = y_pred / (norm_pred[..., tf.newaxis] + eps)

    dot_product = tf.reduce_sum(y_true_norm * y_pred_norm, axis=-1)
    dot_product = tf.clip_by_value(dot_product, -1.0 + eps, 1.0 - eps)

    angle = tf.acos(dot_product)
    masked_angle = angle * mask

    num_valid = tf.reduce_sum(mask) + 1e-8
    return tf.reduce_sum(masked_angle) / num_valid


def spectral_cosine_loss_masked(y_true, y_pred, mask):
    """Cosine loss (1 - cos) with mask. More stable than acos at training start."""
    eps = 1e-7

    norm_true = tf.sqrt(tf.reduce_sum(tf.square(y_true), axis=-1) + eps)
    norm_pred = tf.sqrt(tf.reduce_sum(tf.square(y_pred), axis=-1) + eps)

    dot_product = tf.reduce_sum(y_true * y_pred, axis=-1)
    cosine = dot_product / (norm_true * norm_pred + eps)
    cosine = tf.clip_by_value(cosine, -1.0 + eps, 1.0 - eps)

    loss = (1.0 - cosine)
    masked_loss = loss * mask

    num_valid = tf.reduce_sum(mask) + 1e-8
    return tf.reduce_sum(masked_loss) / num_valid


def combined_loss_masked(y_true, y_pred, mask, config):
    """Combined loss with validity mask (L1 + SAM)."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    mask = tf.cast(mask, tf.float32)

    y_pred = tf.where(tf.math.is_nan(y_pred), tf.zeros_like(y_pred), y_pred)
    y_pred = tf.where(tf.math.is_inf(y_pred), tf.zeros_like(y_pred), y_pred)

    use_band_weights = getattr(config, "USE_BAND_L1_WEIGHTS", False)
    band_weights = getattr(config, "BAND_L1_WEIGHTS", None) if use_band_weights else None
    l1 = l1_loss_masked(y_true, y_pred, mask, band_weights=band_weights)

    sam_angle = spectral_angle_mapper_loss_masked(y_true, y_pred, mask)
    sam_cos = spectral_cosine_loss_masked(y_true, y_pred, mask)
    sam_mode = getattr(config, "SAM_MODE", "acos")
    sam_loss = sam_cos if sam_mode == "cosine" else sam_angle

    l1 = tf.where(tf.math.is_nan(l1), tf.constant(0.5), l1)
    sam_loss = tf.where(tf.math.is_nan(sam_loss), tf.constant(1.0), sam_loss)

    sam_scale = getattr(config, "SAM_LOSS_SCALE", 1.0)
    total = (config.LAMBDA_L1 * l1 +
             config.LAMBDA_SAM * sam_loss * sam_scale)

    valid_ratio = tf.reduce_mean(mask)

    loss_dict = {'l1': l1, 'sam': sam_angle, 'sam_loss': sam_loss, 'valid_ratio': valid_ratio}

    return total, loss_dict

print("Loss functions defined (L1 + SAM)")

## 4. Data Loading and Augmentation

**Data:** TFRecords from Google Cloud Storage (GEE exports)

**Augmentations:**
- Synchronized random crop (tiles 129-130 -> 128x128)
- Horizontal/vertical flip
- 90 degree rotation (0, 90, 180, 270)

**Validation:** Deterministic center crop (no augmentation)

In [ ]:
def parse_tfrecord(example, config):
    """Parse TFRecord exported from GEE with validity mask."""
    feature_description = {
        'B2': tf.io.VarLenFeature(tf.float32),
        'B3': tf.io.VarLenFeature(tf.float32),
        'B4': tf.io.VarLenFeature(tf.float32),
        'B5': tf.io.VarLenFeature(tf.float32),
        'B6': tf.io.VarLenFeature(tf.float32),
        'B7': tf.io.VarLenFeature(tf.float32),
        'B8': tf.io.VarLenFeature(tf.float32),
        'valid': tf.io.VarLenFeature(tf.float32),
    }

    parsed = tf.io.parse_single_example(example, feature_description)

    def find_dimensions(num_values):
        num_values_f = tf.cast(num_values, tf.float32)
        sqrt_n = tf.sqrt(num_values_f)
        sqrt_n_int = tf.cast(sqrt_n, tf.int32)
        is_square = tf.equal(sqrt_n_int * sqrt_n_int, num_values)

        def get_square_dims():
            return sqrt_n_int, sqrt_n_int

        def find_divisor():
            candidate_heights = tf.constant([128, 129, 130, 131, 132, 256, 257, 258, 512], dtype=tf.int32)
            remainders = num_values % candidate_heights
            valid_mask = tf.equal(remainders, 0)
            first_valid_idx = tf.argmax(tf.cast(valid_mask, tf.int32))
            height = candidate_heights[first_valid_idx]
            width = num_values // height
            any_valid = tf.reduce_any(valid_mask)
            height = tf.cond(any_valid, lambda: height, lambda: sqrt_n_int + 1)
            width = tf.cond(any_valid, lambda: width, lambda: num_values // (sqrt_n_int + 1))
            return height, width

        return tf.cond(is_square, get_square_dims, find_divisor)

    def extract_band(band_name):
        dense = tf.sparse.to_dense(parsed[band_name])
        dense = tf.where(tf.math.is_nan(dense), tf.zeros_like(dense), dense)
        num_values = tf.shape(dense)[0]
        height, width = find_dimensions(num_values)
        return tf.reshape(dense, [height, width])

    b2, b3, b4 = extract_band('B2'), extract_band('B3'), extract_band('B4')
    rgb = tf.stack([b4, b3, b2], axis=-1)

    b5, b6, b7, b8 = extract_band('B5'), extract_band('B6'), extract_band('B7'), extract_band('B8')
    nir_rededge = tf.stack([b5, b6, b7, b8], axis=-1)

    valid_mask = extract_band('valid')

    rgb = tf.cast(rgb, tf.float32)
    nir_rededge = tf.cast(nir_rededge, tf.float32)
    valid_mask = tf.cast(valid_mask, tf.float32)

    # Sentinel-2 normalization
    rgb = tf.clip_by_value(rgb, 0.0, 10000.0) / 10000.0
    nir_rededge = tf.clip_by_value(nir_rededge, 0.0, 10000.0) / 10000.0

    # Combined mask: valid AND non-black
    rgb_sum = tf.reduce_sum(rgb, axis=-1)
    nir_sum = tf.reduce_sum(nir_rededge, axis=-1)
    non_black_mask = tf.cast((rgb_sum > 0.01) | (nir_sum > 0.01), tf.float32)
    combined_mask = valid_mask * non_black_mask

    rgb = tf.where(tf.math.is_nan(rgb), tf.zeros_like(rgb), rgb)
    rgb = tf.where(tf.math.is_inf(rgb), tf.zeros_like(rgb), rgb)
    nir_rededge = tf.where(tf.math.is_nan(nir_rededge), tf.zeros_like(nir_rededge), nir_rededge)
    nir_rededge = tf.where(tf.math.is_inf(nir_rededge), tf.zeros_like(nir_rededge), nir_rededge)

    return rgb, nir_rededge, combined_mask


def random_crop_pair_with_mask(rgb, nir, mask, crop_size):
    """Synchronized random crop for RGB, NIR and mask."""
    h = tf.shape(rgb)[0]
    w = tf.shape(rgb)[1]

    max_offset_h = tf.maximum(h - crop_size, 0)
    max_offset_w = tf.maximum(w - crop_size, 0)

    offset_h = tf.random.uniform((), 0, max_offset_h + 1, dtype=tf.int32)
    offset_w = tf.random.uniform((), 0, max_offset_w + 1, dtype=tf.int32)

    rgb_cropped = rgb[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    nir_cropped = nir[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    mask_cropped = mask[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size]

    rgb_cropped = tf.image.resize_with_crop_or_pad(rgb_cropped, crop_size, crop_size)
    nir_cropped = tf.image.resize_with_crop_or_pad(nir_cropped, crop_size, crop_size)
    mask_cropped = tf.image.resize_with_crop_or_pad(mask_cropped[..., tf.newaxis], crop_size, crop_size)[..., 0]

    return rgb_cropped, nir_cropped, mask_cropped


def center_crop_pair_with_mask(rgb, nir, mask, crop_size):
    """Center crop for validation."""
    h = tf.shape(rgb)[0]
    w = tf.shape(rgb)[1]

    offset_h = tf.maximum((h - crop_size) // 2, 0)
    offset_w = tf.maximum((w - crop_size) // 2, 0)

    rgb_cropped = rgb[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    nir_cropped = nir[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    mask_cropped = mask[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size]

    rgb_cropped = tf.image.resize_with_crop_or_pad(rgb_cropped, crop_size, crop_size)
    nir_cropped = tf.image.resize_with_crop_or_pad(nir_cropped, crop_size, crop_size)
    mask_cropped = tf.image.resize_with_crop_or_pad(mask_cropped[..., tf.newaxis], crop_size, crop_size)[..., 0]

    return rgb_cropped, nir_cropped, mask_cropped


def augment_with_mask(rgb, nir, mask, config):
    """Data augmentation with mask."""
    rgb, nir, mask = random_crop_pair_with_mask(rgb, nir, mask, config.PATCH_SIZE)

    if tf.random.uniform(()) > 0.5:
        rgb = tf.image.flip_left_right(rgb)
        nir = tf.image.flip_left_right(nir)
        mask = tf.image.flip_left_right(mask[..., tf.newaxis])[..., 0]

    if tf.random.uniform(()) > 0.5:
        rgb = tf.image.flip_up_down(rgb)
        nir = tf.image.flip_up_down(nir)
        mask = tf.image.flip_up_down(mask[..., tf.newaxis])[..., 0]

    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    rgb = tf.image.rot90(rgb, k)
    nir = tf.image.rot90(nir, k)
    mask = tf.image.rot90(mask[..., tf.newaxis], k)[..., 0]

    return rgb, nir, mask


def prepare_validation_with_mask(rgb, nir, mask, config):
    """Validation preparation with mask."""
    rgb, nir, mask = center_crop_pair_with_mask(rgb, nir, mask, config.PATCH_SIZE)
    return rgb, nir, mask

print("Parsing functions with validity mask defined")

In [ ]:
def load_dataset(pattern, config, is_training=True):
    """Load dataset from GCS pattern with mask."""
    files = tf.io.gfile.glob(pattern)
    print(f"   Found {len(files)} files")

    dataset = tf.data.TFRecordDataset(files, compression_type='GZIP')
    dataset = dataset.map(
        lambda x: parse_tfrecord(x, config),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if is_training:
        dataset = dataset.shuffle(1000)
        dataset = dataset.map(
            lambda rgb, nir, mask: augment_with_mask(rgb, nir, mask, config),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    else:
        dataset = dataset.map(
            lambda rgb, nir, mask: prepare_validation_with_mask(rgb, nir, mask, config),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset


def load_train_val_split(train_pattern, config, val_split=0.1):
    """Load training data and create train/val split with mask."""
    files = tf.io.gfile.glob(train_pattern)
    print(f"   Found {len(files)} training files")

    np.random.seed(42)
    np.random.shuffle(files)

    n_val = max(1, int(len(files) * val_split))
    val_files = files[:n_val]
    train_files = files[n_val:]

    print(f"   Split: {len(train_files)} train, {len(val_files)} val")

    train_ds = tf.data.TFRecordDataset(train_files, compression_type='GZIP')
    train_ds = train_ds.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(1000)
    train_ds = train_ds.map(
        lambda rgb, nir, mask: augment_with_mask(rgb, nir, mask, config),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    train_ds = train_ds.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.TFRecordDataset(val_files, compression_type='GZIP')
    val_ds = val_ds.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.map(
        lambda rgb, nir, mask: prepare_validation_with_mask(rgb, nir, mask, config),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    val_ds = val_ds.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds


def load_all_data_combined(train_pattern, test_pattern, config, val_split=0.1):
    """Combine train + test with mask."""
    train_files = tf.io.gfile.glob(train_pattern)
    test_files = tf.io.gfile.glob(test_pattern)
    all_files = train_files + test_files

    print(f"   Combined: {len(train_files)} train + {len(test_files)} test = {len(all_files)} total")

    np.random.seed(42)
    np.random.shuffle(all_files)

    n_val = max(1, int(len(all_files) * val_split))
    val_files = all_files[:n_val]
    train_files_split = all_files[n_val:]

    print(f"   Split: {len(train_files_split)} train, {len(val_files)} val")

    train_ds = tf.data.TFRecordDataset(train_files_split, compression_type='GZIP')
    train_ds = train_ds.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.shuffle(1000)
    train_ds = train_ds.map(
        lambda rgb, nir, mask: augment_with_mask(rgb, nir, mask, config),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    train_ds = train_ds.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.TFRecordDataset(val_files, compression_type='GZIP')
    val_ds = val_ds.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.map(
        lambda rgb, nir, mask: prepare_validation_with_mask(rgb, nir, mask, config),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    val_ds = val_ds.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds

print("Loading functions with mask defined")

## 5. Trainer Class

Modern deep learning techniques included:

| Technique | Description | Benefit |
|-----------|-------------|--------|
| **Mixed Precision** | fp16 compute, fp32 accumulate | 2x faster, less VRAM |
| **Gradient Clipping** | Limits gradient norm | Prevents gradient explosion |
| **EMA** | Exponential Moving Average of weights | More stable model |
| **Warmup + Cosine** | LR rises then decays in cosine | Optimal convergence |
| **Checkpointing** | Saves complete state | Resume after crash |
| **Early Stopping** | Stops if stagnation | Prevents overfitting |

Note: The EMA model is often **better** than the normal model for inference!

In [ ]:
class MicroNIRTrainer:
    """Complete trainer for MicroNIR with modern deep learning techniques."""

    def __init__(self, config, checkpoint_dir='checkpoints_micronir', steps_per_epoch=None):
        self.config = config
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)

        # Build model
        self.model = build_micronir(config)
        print(f"Model created: {self.model.count_params():,} parameters")

        # EMA (Exponential Moving Average)
        self.ema = EMA(self.model, decay=config.EMA_DECAY) if config.USE_EMA else None
        if self.ema:
            print(f"EMA enabled (decay={config.EMA_DECAY})")

        # Learning Rate Schedule
        if steps_per_epoch:
            total_steps = config.EPOCHS * steps_per_epoch
            warmup_steps = getattr(config, 'WARMUP_STEPS', steps_per_epoch)
            self.lr_schedule = WarmupCosineDecay(
                base_lr=config.LEARNING_RATE,
                warmup_steps=warmup_steps,
                decay_steps=total_steps,
                warmup_min_lr=getattr(config, 'WARMUP_MIN_LR', 1e-7)
            )
            print(f"LR: Warmup {warmup_steps} steps + Cosine decay over {total_steps} steps")
        else:
            self.lr_schedule = config.LEARNING_RATE

        # Optimizer with gradient clipping
        base_optimizer = keras.optimizers.AdamW(
            learning_rate=self.lr_schedule,
            weight_decay=config.WEIGHT_DECAY,
            clipnorm=config.GRADIENT_CLIP_NORM
        )

        # Mixed Precision handling
        if config.USE_MIXED_PRECISION:
            self.optimizer = tf.keras.mixed_precision.LossScaleOptimizer(base_optimizer)
        else:
            self.optimizer = base_optimizer

        print(f"AdamW with Gradient Clipping (norm={config.GRADIENT_CLIP_NORM})")

        # Metrics
        self.train_loss = tf.keras.metrics.Mean()
        self.val_loss = tf.keras.metrics.Mean()
        self.train_valid_ratio = tf.keras.metrics.Mean()
        self.best_val_loss = float('inf')
        self.best_ema_val_loss = float('inf')

        # ReduceLROnPlateau (backup if fixed schedule)
        self.current_lr = config.LEARNING_RATE
        self.patience_lr = 5
        self.factor_lr = 0.5
        self.min_lr = 1e-6
        self.epochs_no_improve = 0

        # Early stopping
        self.patience_stop = 15
        self.epochs_no_improve_stop = 0

        # For resume
        self.start_epoch = 0
        self.global_step = 0
        self.history = {'train_loss': [], 'val_loss': [], 'ema_val_loss': [], 'lr': []}

    def reduce_lr_on_plateau(self, val_loss):
        """Reduce LR if no improvement (backup)."""
        if isinstance(self.lr_schedule, float):
            if val_loss < self.best_val_loss:
                self.epochs_no_improve = 0
            else:
                self.epochs_no_improve += 1
                if self.epochs_no_improve >= self.patience_lr:
                    old_lr = self.current_lr
                    self.current_lr = max(self.current_lr * self.factor_lr, self.min_lr)
                    if self.current_lr < old_lr:
                        inner_opt = self.optimizer.inner_optimizer if hasattr(self.optimizer, 'inner_optimizer') else self.optimizer
                        inner_opt.learning_rate.assign(self.current_lr)
                        print(f"   ReduceLROnPlateau: {old_lr:.2e} -> {self.current_lr:.2e}")
                        self.epochs_no_improve = 0

    def check_early_stopping(self, val_loss):
        """Early stopping check."""
        if val_loss < self.best_val_loss:
            self.epochs_no_improve_stop = 0
        else:
            self.epochs_no_improve_stop += 1

        if self.epochs_no_improve_stop >= self.patience_stop:
            print(f"   Early Stopping after {self.patience_stop} epochs without improvement")
            return True
        return False

    def save_checkpoint(self, epoch):
        """Save complete checkpoint + sync to Google Drive."""
        import json
        import shutil

        checkpoint = {
            'epoch': epoch,
            'global_step': self.global_step,
            'best_val_loss': float(self.best_val_loss),
            'best_ema_val_loss': float(self.best_ema_val_loss),
            'current_lr': float(self.current_lr),
            'epochs_no_improve': self.epochs_no_improve,
            'epochs_no_improve_stop': self.epochs_no_improve_stop,
            'history': self.history
        }

        self.model.save_weights(f'{self.checkpoint_dir}/checkpoint_epoch_{epoch:03d}.weights.h5')
        self.model.save_weights(f'{self.checkpoint_dir}/checkpoint_latest.weights.h5')

        # Save EMA weights
        if self.ema:
            self.ema.apply_to_model()
            self.model.save_weights(f'{self.checkpoint_dir}/checkpoint_ema_latest.weights.h5')
            self.ema.restore_model()

        with open(f'{self.checkpoint_dir}/trainer_state.json', 'w') as f:
            json.dump(checkpoint, f)

        print(f"   Checkpoint saved: epoch {epoch}")

        # Sync to Google Drive
        if getattr(self.config, 'SAVE_TO_DRIVE', False):
            drive_path = getattr(self.config, 'DRIVE_PATH', '/content/drive/MyDrive/MicroNIR_Checkpoints')
            if os.path.exists('/content/drive'):
                try:
                    os.makedirs(drive_path, exist_ok=True)

                    files_to_sync = [
                        f'{self.checkpoint_dir}/checkpoint_latest.weights.h5',
                        f'{self.checkpoint_dir}/checkpoint_ema_latest.weights.h5',
                        f'{self.checkpoint_dir}/trainer_state.json',
                        'micronir_best.weights.h5',
                        'micronir_best_ema.weights.h5'
                    ]

                    for file_path in files_to_sync:
                        if os.path.exists(file_path):
                            shutil.copy2(file_path, drive_path)

                    print(f"   Synced to Drive: {drive_path}")
                except Exception as e:
                    print(f"   Warning: Drive sync error: {e}")

    def load_checkpoint(self, from_drive=False):
        """Load checkpoint for resume."""
        import json
        import os
        import shutil

        # Restore from Google Drive if requested
        if from_drive and os.path.exists('/content/drive'):
            drive_path = getattr(self.config, 'DRIVE_PATH', '/content/drive/MyDrive/MicroNIR_Checkpoints')
            if os.path.exists(drive_path):
                print(f"   Restoring from Drive: {drive_path}")
                os.makedirs(self.checkpoint_dir, exist_ok=True)

                for item in os.listdir(drive_path):
                    src = os.path.join(drive_path, item)
                    if os.path.isfile(src):
                        if 'checkpoint' in item or 'trainer_state' in item:
                            dst = os.path.join(self.checkpoint_dir, item)
                        else:
                            dst = item
                        shutil.copy2(src, dst)
                        print(f"      <- {item}")
                print("   Drive restore complete!")

        checkpoint_path = f'{self.checkpoint_dir}/trainer_state.json'

        if not os.path.exists(checkpoint_path):
            print("   No checkpoint found, starting from scratch")
            return False

        with open(checkpoint_path, 'r') as f:
            checkpoint = json.load(f)

        self.start_epoch = checkpoint['epoch'] + 1
        self.global_step = checkpoint.get('global_step', 0)
        self.best_val_loss = checkpoint['best_val_loss']
        self.best_ema_val_loss = checkpoint.get('best_ema_val_loss', float('inf'))
        self.current_lr = checkpoint['current_lr']
        self.epochs_no_improve = checkpoint['epochs_no_improve']
        self.epochs_no_improve_stop = checkpoint['epochs_no_improve_stop']
        self.history = checkpoint['history']

        # Restore optimizer step for correct LR schedule
        if hasattr(self.optimizer, 'iterations'):
            self.optimizer.iterations.assign(self.global_step)
            print(f"   Optimizer step synced: {self.global_step} (correct LR)")

        weights_path = f'{self.checkpoint_dir}/checkpoint_latest.weights.h5'
        if os.path.exists(weights_path):
            self.model.load_weights(weights_path)

            # Reload EMA if available
            if self.ema:
                ema_path = f'{self.checkpoint_dir}/checkpoint_ema_latest.weights.h5'
                if os.path.exists(ema_path):
                    self.ema.apply_to_model()
                    self.model.load_weights(ema_path)
                    for shadow, weight in zip(self.ema.ema_weights, self.model.weights):
                        shadow.assign(weight)
                    self.model.load_weights(weights_path)

            print(f"   Checkpoint loaded: resuming at epoch {self.start_epoch}")
            print(f"     Best val loss: {self.best_val_loss:.4f}")
            return True

        return False

    @tf.function
    def train_step(self, rgb, nir, mask):
        """Training step with validity mask."""
        with tf.GradientTape() as tape:
            pred = self.model(rgb, training=True)
            loss, loss_dict = combined_loss_masked(nir, pred, mask, self.config)

        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))
        self.train_loss.update_state(loss)
        self.train_valid_ratio.update_state(loss_dict['valid_ratio'])
        return loss_dict

    @tf.function
    def val_step(self, rgb, nir, mask):
        """Validation step with mask."""
        pred = self.model(rgb, training=False)
        loss, loss_dict = combined_loss_masked(nir, pred, mask, self.config)
        self.val_loss.update_state(loss)
        return loss_dict

    def evaluate_ema(self, val_ds):
        """Evaluate EMA model with mask."""
        if not self.ema:
            return None

        self.ema.apply_to_model()

        ema_loss = tf.keras.metrics.Mean()
        for rgb, nir, mask in val_ds:
            pred = self.model(rgb, training=False)
            loss, _ = combined_loss_masked(nir, pred, mask, self.config)
            ema_loss.update_state(loss)

        self.ema.restore_model()
        return ema_loss.result().numpy()

    def train(self, train_ds, val_ds, resume=True, from_drive=False):
        """
        Complete training with all modern techniques.

        Args:
            train_ds: Training dataset
            val_ds: Validation dataset
            resume: If True, resume from last local checkpoint
            from_drive: If True, restore from Google Drive first
        """

        if resume or from_drive:
            self.load_checkpoint(from_drive=from_drive)

        for epoch in range(self.start_epoch, self.config.EPOCHS):
            # Get current LR
            if hasattr(self.lr_schedule, '__call__'):
                current_lr = self.lr_schedule(self.global_step).numpy()
            else:
                current_lr = self.current_lr

            print(f"\n{'='*60}")
            print(f"Epoch {epoch+1}/{self.config.EPOCHS} | LR: {current_lr:.2e}")
            print(f"{'='*60}")

            # Training
            self.train_loss.reset_state()
            self.train_valid_ratio.reset_state()

            for step, (rgb, nir, mask) in enumerate(train_ds):
                loss_dict = self.train_step(rgb, nir, mask)

                # Update EMA after each step
                if self.ema:
                    self.ema.update()

                self.global_step += 1

                if step % 50 == 0:
                    valid_pct = loss_dict['valid_ratio'].numpy() * 100
                    sam_val = loss_dict['sam']
                    print(f"  Step {step} | L1:{loss_dict['l1']:.4f} SAM:{sam_val:.4f} Valid:{valid_pct:.1f}%")

            # Validation of normal model
            self.val_loss.reset_state()
            for rgb, nir, mask in val_ds:
                self.val_step(rgb, nir, mask)

            train_l = self.train_loss.result().numpy()
            val_l = self.val_loss.result().numpy()
            avg_valid = self.train_valid_ratio.result().numpy() * 100

            # Validation of EMA model
            ema_val_l = self.evaluate_ema(val_ds)

            print(f"\nTrain: {train_l:.4f} | Val: {val_l:.4f} | Avg Valid: {avg_valid:.1f}%", end="")
            if ema_val_l:
                print(f" | EMA Val: {ema_val_l:.4f}")
            else:
                print()

            # History
            self.history['train_loss'].append(float(train_l))
            self.history['val_loss'].append(float(val_l))
            self.history['ema_val_loss'].append(float(ema_val_l) if ema_val_l else None)
            self.history['lr'].append(float(current_lr))

            # ReduceLROnPlateau (if no schedule)
            self.reduce_lr_on_plateau(val_l)

            # Save best model (normal) + sync Drive
            if val_l < self.best_val_loss:
                self.best_val_loss = val_l
                self.model.save_weights('micronir_best.weights.h5')
                print(f"   Best model saved (val: {val_l:.4f})")
                self._sync_best_to_drive('micronir_best.weights.h5')

            # Save best EMA model + sync Drive
            if ema_val_l and ema_val_l < self.best_ema_val_loss:
                self.best_ema_val_loss = ema_val_l
                self.ema.apply_to_model()
                self.model.save_weights('micronir_best_ema.weights.h5')
                self.ema.restore_model()
                print(f"   Best EMA model saved (ema_val: {ema_val_l:.4f})")
                self._sync_best_to_drive('micronir_best_ema.weights.h5')

            # Early stopping check
            if self.check_early_stopping(val_l):
                break

            # Checkpoint
            self.save_checkpoint(epoch)

        print("\nTraining complete!")
        print(f"   Best Val Loss: {self.best_val_loss:.4f}")
        if self.best_ema_val_loss < float('inf'):
            print(f"   Best EMA Val Loss: {self.best_ema_val_loss:.4f}")

        return self.history

    def _sync_best_to_drive(self, filename):
        """Sync a best model file to Drive."""
        import shutil
        if getattr(self.config, 'SAVE_TO_DRIVE', False) and os.path.exists('/content/drive'):
            drive_path = getattr(self.config, 'DRIVE_PATH', '/content/drive/MyDrive/MicroNIR_Checkpoints')
            try:
                os.makedirs(drive_path, exist_ok=True)
                if os.path.exists(filename):
                    shutil.copy2(filename, drive_path)
                    print(f"   -> Drive: {filename}")
            except Exception as e:
                print(f"   Warning: sync error: {e}")

print("MicroNIR Trainer defined:")
print("  - MicroNIR + EMA + Warmup Cosine")
print("  - Auto-save to Google Drive")
print("  - Restore from Drive with from_drive=True")
print("")
print("USAGE:")
print("  # First training:")
print("  trainer.train(train_ds, val_ds, resume=False)")
print("")
print("  # Resume after disconnect:")
print("  trainer.train(train_ds, val_ds, resume=True, from_drive=True)")

## 6. Training

### Data Loading Mode

**Mode 1 (SEPARATE)** - Recommended:
- `train/` -> training with augmentation
- `test/` -> independent final evaluation

**Mode 2 (COMBINED)**:
- Merges `train/` + `test/`
- Internal 90/10 split
- No independent test set

In [ ]:
# ============================================================================
# GOOGLE CLOUD AUTHENTICATION (required for GCS access)
# ============================================================================

from google.colab import auth
auth.authenticate_user()
print("Google Cloud authentication successful")

In [ ]:
# ============================================================================
# QUICK DATASET SIZE ESTIMATION
# ============================================================================

import time
start_time = time.time()

print("Quick estimation of example count...")

train_pattern = f"gs://{config.BUCKET}/{config.FOLDER}/train/*.tfrecord.gz"
test_pattern = f"gs://{config.BUCKET}/{config.FOLDER}/test/*.tfrecord.gz"

train_files = tf.io.gfile.glob(train_pattern)
test_files = tf.io.gfile.glob(test_pattern)

print(f"\nFiles found:")
print(f"   Train: {len(train_files)} files")
print(f"   Test: {len(test_files)} files")

# Sample 5 files to estimate examples per file
print(f"\nSampling 5 files for estimation...")
sample_files = train_files[:5]
sample_counts = []

for i, f in enumerate(sample_files):
    ds = tf.data.TFRecordDataset([f], compression_type='GZIP')
    count = sum(1 for _ in ds)
    sample_counts.append(count)
    print(f"   File {i+1}: {count} examples")

avg_per_file = sum(sample_counts) / len(sample_counts)
total_estimated_train = int(avg_per_file * len(train_files))
total_estimated_test = int(avg_per_file * len(test_files))

elapsed = time.time() - start_time

print(f"\n{'='*60}")
print(f"DATASET ESTIMATION:")
print(f"{'='*60}")
print(f"   Examples per file (average): ~{avg_per_file:.0f}")
print(f"   Train estimated: ~{total_estimated_train:,} examples")
print(f"   Test estimated:  ~{total_estimated_test:,} examples")
print(f"   Total estimated: ~{total_estimated_train + total_estimated_test:,} examples")
print(f"{'='*60}")

# Calculate steps_per_epoch
train_after_split = int(total_estimated_train * 0.9)
REAL_STEPS_PER_EPOCH = train_after_split // config.BATCH_SIZE
REAL_VAL_STEPS = (total_estimated_train - train_after_split) // config.BATCH_SIZE

print(f"\nRECOMMENDED CONFIGURATION:")
print(f"   steps_per_epoch: {REAL_STEPS_PER_EPOCH}")
print(f"   validation_steps: {REAL_VAL_STEPS}")
print(f"   Estimation time: {elapsed:.1f}s")

# Save for later use
DATASET_STATS = {
    'train_count': total_estimated_train,
    'test_count': total_estimated_test,
    'steps_per_epoch': REAL_STEPS_PER_EPOCH,
    'val_steps': REAL_VAL_STEPS,
    'avg_per_file': avg_per_file
}

print(f"\nNOTE: These values are ESTIMATES based on a sample.")

In [ ]:
# ============================================================================
# DATA LOADING
# ============================================================================

# Mode choice: False = SEPARATE (recommended), True = COMBINED
USE_ALL_DATA = False

print("\nLoading datasets...")

if USE_ALL_DATA:
    print("   Mode: COMBINED (train + test)")
    train_ds, val_ds = load_all_data_combined(
        f"gs://{config.BUCKET}/{config.FOLDER}/train/*.tfrecord.gz",
        f"gs://{config.BUCKET}/{config.FOLDER}/test/*.tfrecord.gz",
        config,
        val_split=0.1
    )
    test_ds = None
else:
    print("   Mode: SEPARATE (train split + test)")
    train_ds, val_ds = load_train_val_split(
        f"gs://{config.BUCKET}/{config.FOLDER}/train/*.tfrecord.gz",
        config,
        val_split=0.1
    )
    test_ds = load_dataset(
        f"gs://{config.BUCKET}/{config.FOLDER}/test/*.tfrecord.gz",
        config,
        is_training=False
    )

print("\nDatasets loaded")

In [ ]:
# ============================================================================
# COMPLETE DIAGNOSTIC WITH MASK (RUN BEFORE TRAINING!)
# ============================================================================

print("COMPLETE PIPELINE DIAGNOSTIC WITH MASK")
print("="*60)

# ============================================================================
# Test: Check output mean BEFORE any training
# ============================================================================
print("\nUntrained model output test")
print("-"*60)

test_model_diag = build_micronir(config)

# Test with uniform random input
rgb_random = tf.random.uniform((1, 128, 128, 3), 0.0, 0.5)
out_random = test_model_diag(rgb_random, training=False)
print(f"   Input random [0, 0.5]:  mean={tf.reduce_mean(rgb_random):.4f}")
print(f"   Output:                 mean={tf.reduce_mean(out_random):.4f}")

# Test with realistic input (like our RGB data ~0.03-0.1)
rgb_realistic = tf.random.uniform((1, 128, 128, 3), 0.01, 0.1)
out_realistic = test_model_diag(rgb_realistic, training=False)
print(f"   Input realistic [0.01, 0.1]: mean={tf.reduce_mean(rgb_realistic):.4f}")
print(f"   Output:                     mean={tf.reduce_mean(out_realistic):.4f}")

# Verdict
out_mean = tf.reduce_mean(out_realistic).numpy()
if abs(out_mean) < 0.01:
    print(f"\n   PROBLEM: Output ~ 0 ({out_mean:.6f})")
elif 0.05 < out_mean < 0.4:
    print(f"\n   OUTPUT OK: mean={out_mean:.4f} (in good range!)")
else:
    print(f"\n   Unexpected output: mean={out_mean:.4f}")

del test_model_diag

# ============================================================================
# Complete diagnostic with real data
# ============================================================================
print("\n" + "="*60)
print("DIAGNOSTIC WITH REAL DATA")
print("="*60)

# Take a batch for diagnostic
for rgb_batch, nir_batch, mask_batch in train_ds.take(1):
    print(f"\nSHAPES:")
    print(f"   RGB:  {rgb_batch.shape} (dtype: {rgb_batch.dtype})")
    print(f"   NIR:  {nir_batch.shape} (dtype: {nir_batch.dtype})")
    print(f"   Mask: {mask_batch.shape} (dtype: {mask_batch.dtype})")

    print(f"\nMASK STATISTICS:")
    mask_sum = tf.reduce_sum(mask_batch).numpy()
    mask_total = tf.reduce_prod(tf.cast(tf.shape(mask_batch), tf.float32)).numpy()
    mask_ratio = mask_sum / mask_total * 100
    print(f"   Valid pixels: {mask_sum:.0f}/{mask_total:.0f} ({mask_ratio:.1f}%)")

    print(f"\nRGB STATISTICS (input):")
    rgb_min = tf.reduce_min(rgb_batch).numpy()
    rgb_max = tf.reduce_max(rgb_batch).numpy()
    rgb_mean = tf.reduce_mean(rgb_batch).numpy()
    print(f"   Min: {rgb_min:.6f}")
    print(f"   Max: {rgb_max:.6f}")
    print(f"   Mean: {rgb_mean:.6f}")
    print(f"   NaN: {tf.reduce_any(tf.math.is_nan(rgb_batch)).numpy()}")

    print(f"\nNIR STATISTICS (target):")
    nir_min = tf.reduce_min(nir_batch).numpy()
    nir_max = tf.reduce_max(nir_batch).numpy()
    nir_mean = tf.reduce_mean(nir_batch).numpy()
    print(f"   Min: {nir_min:.6f}")
    print(f"   Max: {nir_max:.6f}")
    print(f"   Mean: {nir_mean:.6f}")

    # Test model ON REAL DATA
    print(f"\nMODEL TEST ON REAL DATA:")
    test_model = build_micronir(config)
    print(f"   Parameters: {test_model.count_params():,}")

    test_output = test_model(rgb_batch[:1], training=False)
    out_min = tf.reduce_min(test_output).numpy()
    out_max = tf.reduce_max(test_output).numpy()
    out_mean = tf.reduce_mean(test_output).numpy()
    out_std = tf.math.reduce_std(test_output).numpy()

    print(f"   Output min: {out_min:.6f}")
    print(f"   Output max: {out_max:.6f}")
    print(f"   Output mean: {out_mean:.6f}")
    print(f"   Output std: {out_std:.6f}")

    # Test loss WITH MASK
    print(f"\nLOSS TEST WITH MASK:")
    test_loss, loss_dict = combined_loss_masked(nir_batch[:1], test_output, mask_batch[:1], config)
    print(f"   Total: {test_loss.numpy():.6f}")
    for name, value in loss_dict.items():
        val = value.numpy()
        if name == 'valid_ratio':
            print(f"   {name}: {val*100:.1f}%")
        else:
            print(f"   {name}: {val:.6f}")

    del test_model

print("\n" + "="*60)
print("DIAGNOSTIC COMPLETE")
print("="*60)

In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

# Use REAL example count if available
if 'DATASET_STATS' in dir():
    steps_per_epoch = DATASET_STATS['steps_per_epoch']
    print(f"Using real count: {steps_per_epoch} steps/epoch")
else:
    train_files = tf.io.gfile.glob(f"gs://{config.BUCKET}/{config.FOLDER}/train/*.tfrecord.gz")
    n_train_files = len(train_files)
    estimated_examples = n_train_files * 495  # ~495 examples/file from counting
    steps_per_epoch = max(1, int(estimated_examples * 0.9) // config.BATCH_SIZE)
    print(f"Estimated steps_per_epoch: {steps_per_epoch}")

# Warmup steps
if hasattr(config, 'WARMUP_STEPS'):
    warmup_steps = config.WARMUP_STEPS
else:
    warmup_steps = getattr(config, 'WARMUP_EPOCHS', 1) * steps_per_epoch

print(f"\nTraining configuration:")
print(f"   steps_per_epoch: {steps_per_epoch}")
print(f"   warmup_steps: {warmup_steps}")
print(f"   total_steps: {config.EPOCHS * steps_per_epoch}")

print("\nTraining MicroNIR...")
trainer = MicroNIRTrainer(
    config,
    checkpoint_dir='checkpoints_micronir',
    steps_per_epoch=steps_per_epoch
)

# Patch EMA methods compatibility
if trainer.ema:
    print("Patching EMA methods (apply -> apply_to_model)")
    trainer.ema.apply = trainer.ema.apply_to_model
    trainer.ema.restore = trainer.ema.restore_model

trainer.model.summary()

# resume=True: auto-resume if checkpoint exists
# from_drive=True: restore from Google Drive (essential after crash/disconnect)
print("Starting training (resume from Drive if available)...")
trainer.train(train_ds, val_ds, resume=True, from_drive=True)

## 7. Final Evaluation on Test Set

In [ ]:
if 'test_ds' in globals() and test_ds is not None:
    print("\nFinal evaluation on TEST set...")

    eval_model = trainer.model
    eval_config = trainer.config

    # Safety: Re-batch test_ds if needed
    current_batch_size = getattr(eval_config, 'BATCH_SIZE', 32)
    print(f"   Inference Batch Size: {current_batch_size}")

    test_ds_eval = test_ds.unbatch().batch(current_batch_size).prefetch(tf.data.AUTOTUNE)

    test_loss = tf.keras.metrics.Mean()
    test_valid = tf.keras.metrics.Mean()

    # Float64 accumulators for precision
    mae_sum = tf.zeros((4,), dtype=tf.float64)
    mse_sum = tf.zeros((4,), dtype=tf.float64)
    sam_sum = tf.constant(0.0, dtype=tf.float64)
    ndvi_sum = tf.constant(0.0, dtype=tf.float64)
    ndre_sum = tf.constant(0.0, dtype=tf.float64)
    valid_sum = tf.constant(0.0, dtype=tf.float64)
    ndvi_valid = tf.constant(0.0, dtype=tf.float64)
    ndre_valid = tf.constant(0.0, dtype=tf.float64)
    eps = tf.constant(1e-7, dtype=tf.float32)

    # MRAE accumulator
    mrae_sum = tf.zeros((4,), dtype=tf.float64)
    mrae_valid = tf.constant(0.0, dtype=tf.float64)

    print("   Computing metrics...")
    progbar = tf.keras.utils.Progbar(None)

    for step, (rgb, nir, mask) in enumerate(test_ds_eval):
        # Inference (Training=False)
        pred = eval_model(rgb, training=False)
        # Cast pred to float32 for metric calculations to match nir
        pred = tf.cast(pred, tf.float32)

        # Loss
        loss, loss_dict = combined_loss_masked(nir, pred, mask, eval_config)
        test_loss.update_state(loss)
        test_valid.update_state(loss_dict['valid_ratio'])

        # Pixel-wise metric calculations
        mask_f = tf.cast(mask, tf.float32)
        mask4 = mask_f[..., tf.newaxis]
        valid_pixels = tf.reduce_sum(mask_f)

        # MAE & MSE
        diff = tf.abs(pred - nir) * mask4
        mae_sum += tf.cast(tf.reduce_sum(diff, axis=[0, 1, 2]), tf.float64)

        sq = tf.square(pred - nir) * mask4
        mse_sum += tf.cast(tf.reduce_sum(sq, axis=[0, 1, 2]), tf.float64)

        # MRAE (Mean Relative Absolute Error)
        nir_safe = tf.maximum(tf.abs(nir), 0.01)  # Avoid division by ~0
        relative_error = tf.abs(pred - nir) / nir_safe * mask4
        mrae_sum += tf.cast(tf.reduce_sum(relative_error, axis=[0, 1, 2]), tf.float64)
        mrae_valid += tf.cast(valid_pixels, tf.float64)

        # SAM (Spectral Angle Mapper)
        norm_true = tf.sqrt(tf.reduce_sum(tf.square(nir), axis=-1) + eps)
        norm_pred = tf.sqrt(tf.reduce_sum(tf.square(pred), axis=-1) + eps)
        dot = tf.reduce_sum(nir * pred, axis=-1) / (norm_true * norm_pred + eps)
        dot = tf.clip_by_value(dot, -1.0 + eps, 1.0 - eps)
        angle = tf.acos(dot)
        sam_sum += tf.cast(tf.reduce_sum(angle * mask_f), tf.float64)

        # Spectral indices (NDVI, NDRE)
        pred_clip = tf.clip_by_value(pred, eval_config.OUTPUT_MIN, eval_config.OUTPUT_MAX)
        nir_clip = tf.clip_by_value(nir, eval_config.OUTPUT_MIN, eval_config.OUTPUT_MAX)
        red = tf.clip_by_value(rgb[..., 0], 0.0, 1.0)

        # NDVI (B8 - Red) / (B8 + Red)
        nir_true = nir_clip[..., 3]
        nir_pred = pred_clip[..., 3]
        den_ndvi = nir_true + red
        ndvi_mask = tf.cast(den_ndvi > 0.05, tf.float32)
        ndvi_true = (nir_true - red) / (den_ndvi + eps)
        ndvi_pred = (nir_pred - red) / (nir_pred + red + eps)
        ndvi_sum += tf.cast(tf.reduce_sum(tf.abs(ndvi_true - ndvi_pred) * mask_f * ndvi_mask), tf.float64)
        ndvi_valid += tf.cast(tf.reduce_sum(mask_f * ndvi_mask), tf.float64)

        # NDRE (B5 - Red) / (B5 + Red)
        re_true = nir_clip[..., 0]
        re_pred = pred_clip[..., 0]
        den_ndre = nir_true + re_true
        ndre_mask = tf.cast(den_ndre > 0.05, tf.float32)
        ndre_true = (nir_true - re_true) / (den_ndre + eps)
        ndre_pred = (nir_pred - re_pred) / (nir_pred + re_pred + eps)
        ndre_sum += tf.cast(tf.reduce_sum(tf.abs(ndre_true - ndre_pred) * mask_f * ndre_mask), tf.float64)
        ndre_valid += tf.cast(tf.reduce_sum(mask_f * ndre_mask), tf.float64)

        valid_sum += tf.cast(valid_pixels, tf.float64)

        if step % 10 == 0:
            progbar.update(step + 1)

    # Final aggregation
    valid_sum = tf.maximum(valid_sum, tf.constant(1.0, dtype=tf.float64))
    mae_per_band = mae_sum / valid_sum
    rmse_per_band = tf.sqrt(mse_sum / valid_sum)
    overall_mae = tf.reduce_sum(mae_sum) / (valid_sum * 4.0)
    overall_rmse = tf.sqrt(tf.reduce_sum(mse_sum) / (valid_sum * 4.0))
    sam_mean = sam_sum / valid_sum

    mrae_valid = tf.maximum(mrae_valid, tf.constant(1.0, dtype=tf.float64))
    overall_mrae = tf.reduce_sum(mrae_sum) / (mrae_valid * 4.0) * 100.0  # in %

    ndvi_valid = tf.maximum(ndvi_valid, tf.constant(1.0, dtype=tf.float64))
    ndre_valid = tf.maximum(ndre_valid, tf.constant(1.0, dtype=tf.float64))
    ndvi_mae = ndvi_sum / ndvi_valid
    ndre_mae = ndre_sum / ndre_valid

    band_names = ["B5 (RedEdge)", "B6 (RedEdge)", "B7 (RedEdge)", "B8 (NIR)"]

    print(f"\n\n{'='*60}")
    print(f"FINAL RESULTS - MicroNIR")
    print(f"{'='*60}")
    print(f"   Parameters: {trainer.model.count_params():,}")
    print(f"   Test Loss: {test_loss.result():.4f}")
    print(f"   Valid Pixels: {test_valid.result()*100:.1f}%")
    print(f"")
    print(f"   Test Metrics (MicroNIR):")
    print(f"     L1 (MAE):  {float(overall_mae):.4f}")
    print(f"     MRAE:      {float(overall_mrae):.4f}")
    print(f"     RMSE:      {float(overall_rmse):.4f}")
    print(f"     SAM:       {float(sam_mean):.4f} rad")
    print(f"     NDVI MAE:  {float(ndvi_mae):.4f}")
    print(f"     NDRE MAE:  {float(ndre_mae):.4f}")

    print(f"\n   Per-band detail (MAE / RMSE):")
    for i, bn in enumerate(band_names):
        print(f"     - {bn}: {float(mae_per_band[i]):.4f} / {float(rmse_per_band[i]):.4f}")

    print(f"\n   Comparison with other models:")
    print(f"     WaveletMST (1.05M): MAE=0.0223, SAM=0.0538")
    print(f"     CNN_MSGPP  (0.98M): MAE=0.0249, SAM=0.0576")
    print(f"     MST++      (1.72M): MAE=0.0253, SAM=0.0579")
    print(f"     MicroNIR   ({trainer.model.count_params()/1000:.0f}K):  MAE={float(overall_mae):.4f}, SAM={float(sam_mean):.4f}")

else:
    print("\nNo separate Test Set available (USE_ALL_DATA = True)")

print("\nEvaluation complete!")

## 8. Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_predictions(model, dataset, num_samples=3):
    """Visualize some predictions vs ground truth."""

    # Take a batch
    for batch in dataset.take(1):
        if len(batch) == 3:
            rgb_batch, nir_batch, mask_batch = batch
        else:
            rgb_batch, nir_batch = batch
            mask_batch = None

        pred_batch = model(rgb_batch, training=False)

        # Adjust num_samples in case batch is smaller
        actual_samples = min(num_samples, rgb_batch.shape[0])

        fig, axes = plt.subplots(actual_samples, 6, figsize=(18, 3*actual_samples))

        # Handle case where there's only one sample
        if actual_samples == 1:
            axes = axes[np.newaxis, :]

        for i in range(actual_samples):
            rgb = rgb_batch[i].numpy()
            nir_true = nir_batch[i].numpy()
            nir_pred = pred_batch[i].numpy()
            mask = mask_batch[i].numpy() if mask_batch is not None else None

            # RGB
            axes[i, 0].imshow(np.clip(rgb * 3, 0, 1))  # Boost brightness
            axes[i, 0].set_title('RGB Input')
            axes[i, 0].axis('off')

            # B5 (RedEdge 1)
            axes[i, 1].imshow(nir_true[:,:,0], cmap='RdYlGn')
            axes[i, 1].set_title('B5 True')
            axes[i, 1].axis('off')

            axes[i, 2].imshow(nir_pred[:,:,0], cmap='RdYlGn')
            axes[i, 2].set_title('B5 Pred')
            axes[i, 2].axis('off')

            # B8 (NIR)
            axes[i, 3].imshow(nir_true[:,:,3], cmap='RdYlGn')
            axes[i, 3].set_title('B8 True (NIR)')
            axes[i, 3].axis('off')

            axes[i, 4].imshow(nir_pred[:,:,3], cmap='RdYlGn')
            axes[i, 4].set_title('B8 Pred (NIR)')
            axes[i, 4].axis('off')

            # Error (masked if available)
            error = np.abs(nir_true[:,:,3] - nir_pred[:,:,3])
            mae = np.mean(error)
            if mask is not None:
                error = error * mask
                valid_pixels = np.sum(mask)
                if valid_pixels > 0:
                    mae = np.sum(error) / valid_pixels

            im = axes[i, 5].imshow(error, cmap='hot', vmin=0, vmax=0.1)
            axes[i, 5].set_title(f'MAE for B8: {mae:.4f}')
            axes[i, 5].axis('off')

        plt.tight_layout()
        plt.show()

print(f"Using MicroNIR model")

# Visualize on test set (if available)
if 'test_ds' in globals() and test_ds is not None:
    print("Visualizing predictions on test set (RANDOM TILES):")
    random_test_ds = test_ds.unbatch().shuffle(buffer_size=2000).batch(10)
    visualize_predictions(trainer.model, random_test_ds, num_samples=5)
else:
    print("Visualizing predictions on validation set (RANDOM TILES):")
    random_val_ds = val_ds.unbatch().shuffle(buffer_size=2000).batch(10)
    visualize_predictions(trainer.model, random_val_ds, num_samples=5)